In [ ]:
import gymnasium as gym
import numpy as np

In [ ]:
# -------------------------------------------------
# Create FrozenLake Environment
# -------------------------------------------------

env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
env = env.unwrapped

n_states = env.observation_space.n
n_actions = env.action_space.n

gamma = 0.99
theta = 1e-8


In [ ]:
# -------------------------------------------------
# Policy Evaluation
# -------------------------------------------------
# Code here
def policy_evaluation(env, policy, gamma, theta):

    V = np.zeros(env.observation_space.n)

    while True:
        delta = 0
        for state in range(n_states):
            v = V[state]
            new_v = 0
            for action, action_prob in enumerate(policy[state]):
                for prob, next_state, reward, done in env.P[state][action]:
                    new_v += action_prob * prob * (reward + gamma * V[next_state] * (not done))
            V[state] = new_v
            delta = max(delta, abs(v - V[state]))
        if delta < theta:
            break
    return V

In [ ]:
# -------------------------------------------------
# Policy Improvement
# -------------------------------------------------

# Code here
def policy_improvement(env, V, policy, gamma):
    policy_stable = True
    for state in range(n_states):
        old_action = np.argmax(policy[state])
        action_values = np.zeros(n_actions)
        for action in range(n_actions):
            for prob, next_state, reward, done in env.P[state][action]:
                action_values[action] += prob * (reward + gamma * V[next_state] * (not done))
        best_action = np.argmax(action_values)
        policy[state] = np.eye(n_actions)[best_action]
        if old_action != best_action:
            policy_stable = False
    return policy, policy_stable

In [ ]:
# -------------------------------------------------
# Policy Iteration
# -------------------------------------------------

# Code here
def policy_iteration(env, gamma, theta):

    iteration = 0

    policy = np.ones((n_states, n_actions)) / n_actions

    V = np.zeros(n_states)

    while True:
        V = policy_evaluation(env, policy, gamma, theta)
        policy, policy_stable = policy_improvement(env, V, policy, gamma)

        iteration += 1

        if policy_stable:
            break

    return policy, V, iteration

In [ ]:
# ------------------------------------------------ -
# Display Functions
# ------------------------------------------------ -
def print_value_function(V):
    print("\nOptimal State-Value Function:")
    print(np.round(V.reshape(4, 4), 4))

def print_policy(policy):
    action_symbols = {
        0: "←",
        1: "↓",
        2: "→",
        3: "↑"
    }
    best_actions = np.argmax(policy, axis=1)
    policy_grid = np.array(
        [action_symbols[action] for action in best_actions]
    ).reshape(4, 4)
    print("\nOptimal Policy:")
    print(policy_grid)

In [ ]:
# ------------------------------------------------ -
# Run Policy Iteration
# ------------------------------------------------ -
optimal_policy, optimal_value_function, _ = policy_iteration(
    env,
    gamma=gamma,
    theta=theta
)

print("\n Name: Kailash V")
print("Register Number: 212224240067")
print_value_function(optimal_value_function)
print_policy(optimal_policy)

env.close()


 Name: Kailash V
Register Number: 212224240067

Optimal State-Value Function:
[[0.542  0.4988 0.4707 0.4569]
 [0.5585 0.     0.3583 0.    ]
 [0.5918 0.6431 0.6152 0.    ]
 [0.     0.7417 0.8628 0.    ]]

Optimal Policy:
[['←' '↑' '↑' '↑']
 ['←' '←' '←' '←']
 ['↑' '↓' '←' '←']
 ['←' '→' '↓' '←']]
